# Empirical single-series EWS

This notebook mirrors the `examples/01_*` flow for one empirical time series from Zenodo record 7379358.

It is intentionally single-series only: no ROC/AUROC and no forced-vs-null benchmarking.

Workflow:
1. choose one empirical system and notebook options
2. load the selected text file and truncate to the pre-transition segment
3. apply the paper-style rolling window / step settings, plus `microcosm` segmentation when enabled
4. compute `Variance` and `ResKMD`
5. plot the raw series together with the sampled indicator points


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kews" / "__init__.py").is_file())
NOTEBOOK_DIR = REPO_ROOT / "experiments/empirical"
for path in (REPO_ROOT, NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
OUTPUT_DIR = REPO_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from experiments.indicators import rolling_variance


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import kews
from kews.observables import rff, time_delay
from experiments.indicators import window_size as resolve_window_size

from empirical_utils import download_empirical_archive, load_empirical_system


In [ ]:
# Choose one supported empirical system.
SYSTEM_NAME = "blackout"  # microcosm, voice, mitochondria, climate, blackout

# Relative paths are resolved against the notebook directory below.
DATA_DIR = Path("data")

# Match the Science Advances empirical demo settings by default.
USE_PAPER_WINDOW_SETTINGS = True
wL = 0.5  # Only used when USE_PAPER_WINDOW_SETTINGS = False
STEP_SIZE_OVERRIDE = 1  # Only used when USE_PAPER_WINDOW_SETTINGS = False

# `microcosm` is special in the paper: it uses six disjoint analysis windows.
USE_SYSTEM_SEGMENTS = True  # Uses paper-defined disjoint analysis windows when available

# `N_ENSEMBLES` only matters for stochastic observables such as rff.
N_ENSEMBLES = 1  # Number of ensembles for the ResKMD runs

# Plotting / export switches.
SHOW_TREND_LINE = False
SAVE_OUTPUTS = False

DATA_DIR = (NOTEBOOK_DIR / DATA_DIR).resolve() if not DATA_DIR.is_absolute() else DATA_DIR


In [ ]:
# Download / extract the Zenodo archive if needed, then load one system.
archive_root = download_empirical_archive(DATA_DIR)
system = load_empirical_system(SYSTEM_NAME, DATA_DIR)

time = system["time"]
state = system["state"]
display_name = system["display_name"]
time_label = system["time_label"]
state_label = system["state_label"]
transition_time = system["transition_time"]
analysis_segments = [tuple(bounds) for bounds in system.get("analysis_segments", ())]
analysis_segment_labels = list(system.get("analysis_segment_labels", ()))
use_segmented_analysis = bool(USE_SYSTEM_SEGMENTS and analysis_segments)

# Always compute indicators strictly before the transition marker.
if transition_time is None:
    analysis_mask = np.ones_like(time, dtype=bool)
else:
    analysis_mask = time < transition_time

analysis_time = np.asarray(time[analysis_mask], dtype=float)
analysis_state = np.asarray(state[analysis_mask], dtype=float)

# Use the paper's absolute window sizes by default; otherwise `wL` can stay fractional.
if USE_PAPER_WINDOW_SETTINGS:
    wL = int(system["window_size"])
    step_size = int(system["step_size"])
else:
    step_size = int(STEP_SIZE_OVERRIDE)

window_points = resolve_window_size(len(analysis_state), wL)

# `microcosm` is special in the original paper: keep only the P1-P6 windows
# and map them into slices on the already truncated `analysis_time` grid.
analysis_segment_slices = []
if use_segmented_analysis:
    kept_bounds = []
    kept_labels = []
    for idx, bounds in enumerate(analysis_segments):
        seg_mask = (analysis_time >= bounds[0]) & (analysis_time <= bounds[1])
        seg_indices = np.flatnonzero(seg_mask)
        if seg_indices.size == 0:
            continue
        analysis_segment_slices.append(slice(int(seg_indices[0]), int(seg_indices[-1]) + 1))
        kept_bounds.append((float(analysis_time[seg_indices[0]]), float(analysis_time[seg_indices[-1]])))
        if idx < len(analysis_segment_labels):
            kept_labels.append(analysis_segment_labels[idx])
        else:
            kept_labels.append(f"P{idx + 1}")
    analysis_segments = kept_bounds
    analysis_segment_labels = kept_labels
else:
    analysis_segments = []
    analysis_segment_labels = []

print(f"Archive root: {archive_root}")
print(f"Loaded {display_name} from {system['source_file']} with {system['series_length']} points.")
print(f"Using {analysis_state.size} strictly pre-transition points for EWS computation.")
if use_segmented_analysis:
    print(f"Computing EWS on {len(analysis_segment_slices)} paper-defined segments.")
print(f"Rolling window: {window_points} points; step size: {step_size}.")
display(pd.Series({k: v for k, v in system.items() if k not in {'time', 'state'}}))


In [ ]:
# Convert dense rolling indicators into paper-style sampled window endpoints.
def stride_indicator(indicator, data, window_points, step_size):
    """Keep only every `step_size` rolling-window endpoint from a dense indicator curve."""
    values = np.asarray(indicator, dtype=float)
    sparse = np.full_like(values, np.nan, dtype=float)
    sample_indices = np.arange(window_points - 1, len(values), step_size, dtype=int)
    sparse[sample_indices] = values[sample_indices]
    return sparse


# Dispatch the requested ResKMD configuration for each empirical system.
def get_resdmd_kwargs(system_name, seed):
    """Return the per-system ResKMD configuration requested for the empirical notebook."""
    if system_name == "microcosm":
        return {
            "observable": rff(n_features=100, scale=1e-4, seed=seed),
            "rank": 0.5,
        }
    if system_name in {"voice", "blackout"}:
        return {
            "observable": rff(n_features=100, scale=1e-3, seed=seed),
            "rank": 0.9,
        }
    if system_name == "climate":
        return {
            "observable": time_delay(n_delay=20),
        }
    if system_name == "mitochondria":
        return {
            "observable": rff(n_features=100, scale=1e-3, seed=seed),
            "rank": 0.5,
        }
    raise ValueError(f"Unsupported system for ResKMD config: {system_name}")


# Human-readable summary printed before the computation runs.
def describe_resdmd_config(system_name):
    descriptions = {
        "microcosm": "rff(n_features=100, scale=1e-4), rank=0.5",
        "voice": "rff(n_features=100, scale=1e-3), rank=0.9",
        "blackout": "rff(n_features=100, scale=1e-3), rank=0.9",
        "climate": "time_delay(n_delay=20)",
        "mitochondria": "rff(n_features=100, scale=1e-3), rank=0.5",
    }
    return descriptions[system_name]


# Compute indicators either on the full pre-transition series or, for `microcosm`,
# separately on each paper segment and then merge them back onto one time axis.
def compute_ews_metrics(state, wL, window_points, step_size, system_name, segment_slices=None):
    """Compute variance and ResKMD for one empirical single-series trajectory."""
    data = np.asarray(state, dtype=float)

    if segment_slices:
        merged = {
            "var": np.full(data.shape[0], np.nan, dtype=float),
            "resdmd": np.full(data.shape[0], np.nan, dtype=float),
        }
        for segment_slice in segment_slices:
            # In segmented mode, each segment is processed independently so windows do not
            # cross the paper's analysis gaps.
            segment_state = data[segment_slice]
            if segment_state.size < window_points:
                continue
            segment_metrics = compute_ews_metrics(
                segment_state,
                wL,
                window_points,
                step_size,
                system_name,
            )
            for key, indicator in segment_metrics.items():
                merged[key][segment_slice] = np.asarray(indicator, dtype=float)
        return {key: values for key, values in merged.items()}

    var = rolling_variance(data, window=wL)

    # `N_ENSEMBLES` matters only for stochastic observables such as rff.
    resdmd_ens = []
    for seed in range(N_ENSEMBLES):
        resdmd = kews.resdmd(data, 
            window=wL,
            
            **get_resdmd_kwargs(system_name, seed),
        )
        resdmd_ens.append(np.asarray(resdmd, dtype=float))

    resdmd_mean = np.asarray(resdmd_ens, dtype=float).mean(axis=0)

    return {
        "var": stride_indicator(var, data, window_points, step_size),
        "resdmd": stride_indicator(resdmd_mean, data, window_points, step_size),
    }


print(f"ResKMD config: {describe_resdmd_config(SYSTEM_NAME)}")
ews_metrics = compute_ews_metrics(
    analysis_state,
    wL,
    window_points,
    step_size,
    SYSTEM_NAME,
    segment_slices=analysis_segment_slices,
)


In [ ]:
# Use percentile-based limits so a few large residuals do not dominate the axis.
def robust_ylim(values, low=2, high=98, margin=0.05):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return 0.0, 1.0
    y0, y1 = np.nanpercentile(vals, [low, high])
    if y0 == y1:
        y0, y1 = y0 - 0.5, y1 + 0.5
    pad = (y1 - y0) * margin
    return y0 - pad, y1 + pad


# Darker red marks later sampled windows.
def indicator_point_colors(n_points, base_color):
    if n_points <= 0:
        return base_color
    base_rgb = np.array(mcolors.to_rgb(base_color))
    mix = np.linspace(0.2, 1.0, n_points)[:, None]
    return 1.0 - mix * (1.0 - base_rgb)


def style_axis(ax):
    ax.grid(alpha=0.2)

# Plot one shared-x figure so the raw series and both indicators stay aligned in time.
ews_fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
raw_ax, var_ax, resdmd_ax = axes

raw_ax.plot(time, state, color="C0", lw=2)

if use_segmented_analysis:
    # Shade the analysis windows on every panel for the segmented `microcosm` workflow.
    label_y = np.nanpercentile(state, 97)
    for label, (seg_start, seg_end) in zip(analysis_segment_labels, analysis_segments):
        for ax in axes:
            ax.axvspan(seg_start, seg_end, color="0.4", alpha=0.1, lw=0)
        raw_ax.text(seg_end, label_y, label, ha="right", va="bottom", color="black", fontsize=10)

if transition_time is not None:
    # Keep the tipping marker visible on all panels, but only the raw series extends beyond it.
    post_end = np.nanmax(time)
    for ax in axes:
        ax.axvspan(transition_time, post_end, color="0.4", alpha=0.2)
        ax.axvline(transition_time, color="0.4", linestyle="--", lw=2)

raw_ax.set_ylabel(state_label)
style_axis(raw_ax)

for ax, key, ylabel, use_robust in [
    (var_ax, "var", "Variance", False),
    (resdmd_ax, "resdmd", "ResKMD", True),
]:
    # The indicators are only defined on the pre-transition analysis subset.
    values = np.asarray(ews_metrics[key], dtype=float)
    sample_mask = np.isfinite(values)
    n_points = int(sample_mask.sum())
    ax.scatter(
        analysis_time[sample_mask],
        values[sample_mask],
        c=indicator_point_colors(n_points, "C3"),
        s=20,
    )

    if SHOW_TREND_LINE and n_points >= 2:
        # Simple linear overlay to match the paper-style visual summary.
        trend_x = analysis_time[sample_mask]
        trend_coef = np.polyfit(trend_x, values[sample_mask], deg=1)
        trend_y = np.polyval(trend_coef, trend_x)
        ax.plot(trend_x, trend_y, color=indicator_base_color, lw=2)

    ax.set_ylabel(ylabel)
    if use_robust:
        ax.set_ylim(robust_ylim(values))
    style_axis(ax)

for ax in axes:
    ax.xaxis.label.set_size(16)
    ax.yaxis.label.set_size(16)
    ax.tick_params(axis="both", labelsize=13)
    ax.xaxis.get_offset_text().set_fontsize(13)
    ax.yaxis.get_offset_text().set_fontsize(13)
    ax.set_xlabel(time_label)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/empirical_{SYSTEM_NAME}.png', dpi=300, bbox_inches='tight')
plt.show()
